# Validate the MDA Operations Mirror

Confirm that all 15 PostgreSQL tables for shared release `2026.11.03` are present in the Fabric mirrored database with the expected 34,674 total rows. The contract separates 18 realistic hardware components from 2,160 event-participation assignments across 120 lifecycle events. The validation reads the mirror's Delta tables directly from OneLake.

In [ ]:
WORKSPACE_ID = '10327698-2b0d-446f-9b1b-beabe18a4bda'
MIRRORED_DATABASE_ID = 'fdccee22-7557-4d68-a387-aa7fc1d48343'
MIRROR_ROOT = (
    f'abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/'
    f'{MIRRORED_DATABASE_ID}/Tables/mda_ops'
)
EXPECTED_ROWS = {
    'test_event': 120,
    'test_objective': 60,
    'required_feed': 24,
    'site': 40,
    'system_instance': 18,
    'event_participant': 2_160,
    'readiness_history': 12_960,
    'maintenance_action': 108,
    'inventory_position': 144,
    'finding': 2_880,
    'finding_evidence': 8_640,
    'corrective_action': 5_760,
    'report_review': 720,
    'projector_checkpoint': 40,
    'projector_dead_letter': 1_000,
}
RELEASE_ID = '2026.11.03'
print(f'Validating {len(EXPECTED_ROWS)} mirrored tables for release {RELEASE_ID}')

In [ ]:
results = []
for table_name, expected_count in EXPECTED_ROWS.items():
    table_path = f'{MIRROR_ROOT}/{table_name}'
    try:
        actual_count = spark.read.format('delta').load(table_path).count()
        results.append((table_name, expected_count, actual_count, actual_count == expected_count, table_path))
    except Exception as exc:
        results.append((table_name, expected_count, None, False, f'{type(exc).__name__}: {exc}'))

validation_df = spark.createDataFrame(
    results,
    ['source_table', 'expected_rows', 'mirrored_rows', 'passed', 'resolved_table'],
)
display(validation_df.orderBy('source_table'))
failed = validation_df.filter('NOT passed').count()
if failed:
    raise RuntimeError(f'Mirror validation failed for {failed} tables')
print('MDA Operations Mirror row-count contract: OK')